In [72]:
"""
Script de traitement des données de limites planétaires (LP)
Génère un graphique en barres empilées représentant les LP par sous-processus
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings

warnings.filterwarnings('ignore')

In [73]:
# ============================================================================
# 1. PARAMÈTRES GLOBAUX
# ============================================================================

BASE_DIR = r"C:\Users\Arnaud\Documents\PlaneFR\data"
BASE_DATA_DIR = r"C:\Users\Arnaud\Documents\PlaneFR\data\3.8.2"
THRESHOLD_PARAM = "Égalité"  # Peut être changé à "capacité" ou autre

# Récupérer les dossiers de scénarios
SCENARIO_FOLDERS = sorted([d for d in Path(BASE_DATA_DIR).iterdir() if d.is_dir()])
SCENARIO_NAMES = [d.name for d in SCENARIO_FOLDERS]
REFERENCE_SCENARIO_IDX = 0  # "base-year_2015" est le premier

# Paths des fichiers source (communs à tous les scénarios)
FACTEURS_CARAC_FILE = Path(BASE_DIR) / "facteurs de caractérisation.xlsx"
BRIDGE_MATRICES_FILE = Path(BASE_DIR) / "bridge_matrices.xlsx"
SEUILS_FILE = Path(BASE_DIR) / "seuils.xlsx"

# Couleurs pour les 5 catégories de consommation
CATEGORY_COLORS = {
    0: "#1f77b4",  # Bleu
    1: "#ff7f0e",  # Orange
    2: "#2ca02c",  # Vert
    3: "#d62728",  # Rouge
    4: "#9467bd"   # Violet
}

# Fonctions utilitaires pour générer les nuances
def get_dark_shade(hex_color, factor=0.7):
    """Retourne une nuance foncée d'une couleur hex"""
    h = hex_color.lstrip('#')
    rgb = tuple(int(h[i:i+2], 16) for i in (0, 2, 4))
    dark_rgb = tuple(int(c * factor) for c in rgb)
    return '#{:02x}{:02x}{:02x}'.format(*dark_rgb)

def get_light_shade(hex_color, factor=1.3):
    """Retourne une nuance claire d'une couleur hex"""
    h = hex_color.lstrip('#')
    rgb = tuple(int(h[i:i+2], 16) for i in (0, 2, 4))
    light_rgb = tuple(min(255, int(c * factor)) for c in rgb)
    return '#{:02x}{:02x}{:02x}'.format(*light_rgb)


In [74]:
# ============================================================================
# 2. FONCTIONS DE CHARGEMENT DES DONNÉES
# ============================================================================

def load_facteurs_carac():
    """
    Charge le fichier des facteurs de caractérisation.
    
    Returns:
        pd.DataFrame: DataFrame contenant les colonnes:
            - "Sous-processus"
            - "processus du système Terre" (LP)
            - "extensions exiobase"
            - "facteur de caractérisation"
    """
    print("Chargement des facteurs de caractérisation...")
    df = pd.read_excel(FACTEURS_CARAC_FILE)
    print(f"  ✓ {len(df)} lignes chargées")
    return df


def load_bridge_matrices():
    """
    Charge la matrice bridge "Final conso. cat".
    
    Returns:
        pd.DataFrame: Matrice contenant les catégories de consommation
    """
    print("Chargement de la matrice bridge...")
    df = pd.read_excel(BRIDGE_MATRICES_FILE, sheet_name="Final conso. cat.")
    print(f"  ✓ Matrice de shape {df.shape}")
    return df


def load_seuils():
    """
    Charge le fichier des seuils.
    
    Returns:
        pd.DataFrame: DataFrame "synthèse" contenant:
            - Index: noms des LP (et lignes spéciales comme "nom LP", "correspondances", THRESHOLD_PARAM)
            - Colonnes: sous-processus
    """
    print("Chargement des seuils...")
    df = pd.read_excel(SEUILS_FILE, sheet_name="Synthèse", index_col=0)
    print(f"  ✓ {len(df)} lignes chargées")
    return df


def load_d_cba(scenario_folder_path, lp_name, origin="imp"):
    """
    Charge un fichier d_cba (demandes finales) pour un scénario spécifique.
    
    Args:
        scenario_folder_path (Path): Chemin du dossier du scénario
        lp_name (str): Nom du processus du système Terre (ex: "Eau", "Énergie")
        origin (str): "imp" pour importé, "dom" pour domestique
    
    Returns:
        pd.DataFrame: DataFrame des demandes finales
    """
    if origin == "imp":
        folder_name = f"imp_{lp_name}"
    else:
        folder_name = f"dom_{lp_name}"
    
    file_path = Path(scenario_folder_path) / "extensions" / folder_name / "d_cba_k.pkl"
    
    if not file_path.exists():
        print(f"  ⚠ Fichier non trouvé: {file_path}")
        return None
    
    df = pd.read_pickle(file_path)
    return df

In [75]:
# ============================================================================
# 3. FONCTIONS DE TRAITEMENT DES DONNÉES
# ============================================================================

def get_unique_subprocesses(facteurs_carac_df):
    """
    Extrait les sous-processus uniques et crée un mapping vers les LP.
    
    Args:
        facteurs_carac_df (pd.DataFrame): DataFrame des facteurs
    
    Returns:
        dict: Dictionnaire {sous-processus: LP}
    """
    subprocess_to_lp = {}
    for _, row in facteurs_carac_df.iterrows():
        subprocess = row["Sous-processus"]
        lp = row["Processus du système Terre"]
        if subprocess not in subprocess_to_lp:
            subprocess_to_lp[subprocess] = lp
    
    return subprocess_to_lp


def process_scenario(scenario_folder_path, facteurs_carac_df, bridge_matrices_df, seuils_df):
    """
    Traite un scénario complet : charge et agrège les données pour tous les sous-processus.
    
    Args:
        scenario_folder_path (Path): Chemin du dossier du scénario
        facteurs_carac_df (pd.DataFrame): DataFrame des facteurs
        bridge_matrices_df (pd.DataFrame): Matrice bridge
        seuils_df (pd.DataFrame): DataFrame des seuils
    
    Returns:
        tuple: (
            data_by_subprocess (dict): {subprocess_name: traitement_result},
            subprocess_to_lp (dict): Mapping subprocess -> LP
        )
    """
    print(f"  Traitement du scénario : {scenario_folder_path.name}")
    
    subprocess_to_lp = get_unique_subprocesses(facteurs_carac_df)
    data_by_subprocess = {}
    
    for subprocess_name, lp_name in subprocess_to_lp.items():
        result = process_single_subprocess_scenario(subprocess_name, lp_name, 
                                                    scenario_folder_path,
                                                    facteurs_carac_df, bridge_matrices_df, seuils_df)
        if result is not None:
            data_by_subprocess[subprocess_name] = result
    
    return data_by_subprocess, subprocess_to_lp


def process_single_subprocess_scenario(subprocess_name, lp_name, scenario_folder_path,
                                       facteurs_carac_df, bridge_matrices_df, seuils_df):
    """
    Traite un sous-processus spécifique pour un scénario donné.
    [Même logique que process_single_subprocess mais adapté au scenario_folder_path]
    """
    # Étape 1: Charger les fichiers d_cba
    d_cba_imp = load_d_cba(scenario_folder_path, lp_name, origin="imp")
    d_cba_dom = load_d_cba(scenario_folder_path, lp_name, origin="dom")
    
    if d_cba_imp is None or d_cba_dom is None:
        return None
    
    # Étape 2: Récupérer les extensions associées à ce sous-processus
    mask = facteurs_carac_df["Sous-processus"] == subprocess_name
    subprocess_data = facteurs_carac_df[mask]
    
    extensions = subprocess_data["Extensions exiobase"].values
    factors = subprocess_data["Facteurs de caractérisation"].values
    
    # Étape 3: Créer un dictionnaire {extension: facteur}
    extension_factor_map = dict(zip(extensions, factors))
    
    # Étape 4: Filtrer et pondérer les données d_cba
    def filter_and_weight_dcba(d_cba_df, extension_factor_map):
        """Filtre les lignes et applique les pondérations"""
        weighted_rows = []

        for idx, row in d_cba_df.iterrows():
            idx_value = idx[0] if isinstance(idx, tuple) else idx
            
            if idx_value in extension_factor_map:
                factor = extension_factor_map[idx_value]
                weighted_rows.append(row * factor)
        
        if not weighted_rows:
            return pd.Series(dtype=float)
        
        summed = pd.DataFrame(weighted_rows).sum(axis=0)
        
        if isinstance(summed.index, pd.MultiIndex):
            summed.index = summed.index.get_level_values(-1)
        
        return summed
    
    vector_imp = filter_and_weight_dcba(d_cba_imp, extension_factor_map)
    vector_dom = filter_and_weight_dcba(d_cba_dom, extension_factor_map)
    
    if vector_imp.empty or vector_dom.empty:
        return None
    
    # Étape 5: Multiplication matricielle avec la matrice bridge
    bridge_categories = bridge_matrices_df.iloc[1:, 6:]
    bridge_categories.index = bridge_matrices_df.iloc[1:, 5].values
    result_imp = bridge_categories.T @ vector_imp
    result_dom = bridge_categories.T @ vector_dom
    
    # Étape 6: Normalisation par la valeur de conversion
    try:
        if "Conversion" in seuils_df.index and subprocess_name in seuils_df.columns:
            conversion_factor = seuils_df.loc["Conversion", subprocess_name]
            if conversion_factor != 0 and not pd.isna(conversion_factor):
                result_imp = result_imp / conversion_factor
                result_dom = result_dom / conversion_factor
    except Exception:
        pass
    
    return {
        'domestique': result_dom,
        'importé': result_imp,
        'categories': result_imp.index.tolist()
    }


In [76]:
# ============================================================================
# 4. FONCTION DE VISUALISATION
# ============================================================================

def create_stacked_bar_chart(data_by_subprocess, seuils_df, subprocess_to_lp, 
                              scenario_name="", reference_data=None, ax=None, show_legend=True):
    """
    Crée un graphique en barres empilées représentant les LP en pourcentages.
    
    Args:
        data_by_subprocess (dict): {subprocess_name: traitement_result}
        seuils_df (pd.DataFrame): DataFrame des seuils
        subprocess_to_lp (dict): Mapping subprocess -> LP
        scenario_name (str): Nom du scénario pour le titre
        reference_data (dict): Données de référence pour calculer les variations relatives
        ax (matplotlib axis): Axe existant pour tracer. Si None, crée une nouvelle figure
        show_legend (bool): Affiche la légende locale
    
    Returns:
        tuple: (fig, ax) si ax=None, sinon (None, ax)
    """
    # Préparer les données pour le graphique
    subprocesses = list(data_by_subprocess.keys())
    n_subprocesses = len(subprocesses)
    
    # Récupérer les catégories (supposé identique pour tous)
    first_valid_data = None
    for data in data_by_subprocess.values():
        if data is not None:
            first_valid_data = data
            break
    
    if first_valid_data is None:
        print("⚠ Aucune donnée valide à visualiser")
        return None if ax is None else (None, ax)
    
    categories = first_valid_data['categories']
    
    # Initialiser les données pour le graphique
    x_labels = [sp for sp in subprocesses]
    
    # Créer une figure si nécessaire
    if ax is None:
        fig, ax = plt.subplots(figsize=(14, 8))
    else:
        fig = None
    
    # Préparer les hauteurs cumulées pour chaque barre
    bar_width = 0.6
    x_pos = np.arange(n_subprocesses)
    
    bottom_heights = np.zeros(n_subprocesses)
    total_values = np.zeros(n_subprocesses)  # Stocker les valeurs totales absolues
    
    # Itérer sur les catégories et origines - PREMIÈRE PASSE (calcul des totaux)
    for cat_idx, category in enumerate(categories):
        # Part domestique
        heights_dom = []
        for sp in subprocesses:
            data = data_by_subprocess[sp]
            if data is not None and 'domestique' in data:
                heights_dom.append(data['domestique'].get(category, 0))
            else:
                heights_dom.append(0)
        
        heights_dom = np.array(heights_dom)
        
        # Part importée
        heights_imp = []
        for sp in subprocesses:
            data = data_by_subprocess[sp]
            if data is not None and 'importé' in data:
                heights_imp.append(data['importé'].get(category, 0))
            else:
                heights_imp.append(0)
        
        heights_imp = np.array(heights_imp)
        
        # Accumuler les valeurs totales (pour la normalisation)
        total_values += (heights_dom + heights_imp)
    
    # Réinitialiser bottom_heights pour construire le graphique normalisé
    bottom_heights = np.zeros(n_subprocesses)
    
    # Itérer de nouveau pour tracer les barres normalisées - DEUXIÈME PASSE
    for cat_idx, category in enumerate(categories):
        # Part domestique
        heights_dom = []
        for sp in subprocesses:
            data = data_by_subprocess[sp]
            if data is not None and 'domestique' in data:
                heights_dom.append(data['domestique'].get(category, 0))
            else:
                heights_dom.append(0)
        
        heights_dom = np.array(heights_dom)
        # Normaliser en pourcentage
        heights_dom_pct = np.where(total_values > 0, (heights_dom / total_values) * 100, 0)
        
        color_dark = get_dark_shade(CATEGORY_COLORS[cat_idx % len(CATEGORY_COLORS)])
        
        ax.bar(x_pos, heights_dom_pct, bar_width, label=f"{category} - Domestique",
               bottom=bottom_heights, color=color_dark, edgecolor='white', linewidth=0.5)
        
        bottom_heights += heights_dom_pct
        
        # Part importée
        heights_imp = []
        for sp in subprocesses:
            data = data_by_subprocess[sp]
            if data is not None and 'importé' in data:
                heights_imp.append(data['importé'].get(category, 0))
            else:
                heights_imp.append(0)
        
        heights_imp = np.array(heights_imp)
        # Normaliser en pourcentage
        heights_imp_pct = np.where(total_values > 0, (heights_imp / total_values) * 100, 0)
        
        color_light = get_light_shade(CATEGORY_COLORS[cat_idx % len(CATEGORY_COLORS)])
        
        ax.bar(x_pos, heights_imp_pct, bar_width, label=f"{category} - Importé",
               bottom=bottom_heights, color=color_light, edgecolor='white', linewidth=0.5)
        
        bottom_heights += heights_imp_pct
    
    # Ajouter les valeurs totales et variations relatives
    for idx, total_val in enumerate(total_values):
        sp = subprocesses[idx]
        lp = subprocess_to_lp[sp]
        
        # Récupérer l'unité d'affichage pour ce sous-processus
        unit_text = ""
        try:
            if "Unité affichage" in seuils_df.index and sp in seuils_df.columns:
                unit_val = seuils_df.loc["Unité affichage", sp]
                if unit_val is not None and str(unit_val).lower() != "nan":
                    unit_text = f" {unit_val}"
        except (KeyError, TypeError):
            pass
        
        # Afficher le total en gras avec l'unité
        ax.text(
            idx, 102, f"{total_val:.0f}{unit_text}",
            ha='center', va='bottom', fontweight='bold', fontsize=9
        )

        # Afficher la variation relative si données de référence fournies
        if reference_data is not None and sp in reference_data:
            ref_total = (reference_data[sp]['domestique'].sum() + 
                        reference_data[sp]['importé'].sum())
            if ref_total > 0:
                variation_pct = ((total_val - ref_total) / ref_total) * 100
                if variation_pct > 0:
                    text_var = f'+{variation_pct:.0f}%'
                else:
                    text_var = f'{variation_pct:.0f}%'
                
                ax.text(idx, 97, text_var, ha='center', va='bottom', 
                       fontweight='bold', fontsize=8, color='black')

        # Afficher le dépassement (overshoot/undershoot) relatif au seuil
        try:
            if THRESHOLD_PARAM in seuils_df.index and sp in seuils_df.columns:
                threshold = seuils_df.loc[THRESHOLD_PARAM, sp]
                if threshold > 0:
                    overshoot_pct = ((total_val - threshold) / threshold) * 100
                    if overshoot_pct > 0:
                        color = 'red'
                        label_text = f'+{overshoot_pct:.0f}%'
                    else:
                        color = 'green'
                        label_text = f'{overshoot_pct:.0f}%'
                    
                    y_pos = 95 if reference_data is not None else 98
                    ax.text(idx, y_pos, label_text, ha='center', va='top', 
                           fontweight='bold', fontsize=8, color=color)
        except (KeyError, TypeError):
            pass
    
    # Mise en forme du graphique
    ax.set_xlabel('Empreintes environnementales', fontsize=10, fontweight='bold')
    ax.set_ylabel(f"Part de l'empreinte (%)", fontsize=10, fontweight='bold')
    
    if scenario_name:
        ax.set_title(f"{scenario_name}", fontsize=12, fontweight='bold', pad=10)
    
    ax.set_xticks(x_pos)
    ax.set_xticklabels(x_labels, rotation=45, ha='right', fontsize=9)
    ax.set_ylim(0, 110)
    ax.grid(axis='y', alpha=0.3, linestyle='--')
    
    # Légende (si demandée)
    if show_legend:
        handles, labels = ax.get_legend_handles_labels()
        by_label = dict(zip(labels, handles))
        ax.legend(by_label.values(), by_label.keys(), loc='upper left', 
                 bbox_to_anchor=(1.05, 1), fontsize=8)
    
    if fig is not None:
        plt.tight_layout()
    
    return (fig, ax) if fig is not None else (None, ax)

In [77]:

# ============================================================================
# 5. FONCTION DE SYNTHÈSE (MULTI-SCÉNARIOS)
# ============================================================================

def create_synthesis_figure(all_scenarios_data, seuils_df, subprocess_to_lp_list, scenario_names):
    """
    Crée une figure de synthèse avec les 6 scénarios en grille 3x2.
    
    Args:
        all_scenarios_data (list): Liste des dict {subprocess_name: traitement_result} pour chaque scénario
        seuils_df (pd.DataFrame): DataFrame des seuils
        subprocess_to_lp_list (list): Listes des mappings subprocess -> LP
        scenario_names (list): Noms des scénarios
    
    Returns:
        tuple: (fig, axes_list)
    """
    fig = plt.figure(figsize=(18, 12))
    gs = fig.add_gridspec(3, 2, hspace=0.4, wspace=0.3)
    
    reference_idx = REFERENCE_SCENARIO_IDX
    reference_data = all_scenarios_data[reference_idx]
    
    # Positions des subplots
    positions = [
        (0, 0), (0, 1),
        (1, 0), (1, 1),
        (2, 0), (2, 1)
    ]
    
    axes_list = []
    all_handles = None
    all_labels = None
    
    for scenario_idx, (row, col) in enumerate(positions):
        ax = fig.add_subplot(gs[row, col])
        axes_list.append(ax)
        
        scenario_data = all_scenarios_data[scenario_idx]
        scenario_name = scenario_names[scenario_idx]
        subprocess_to_lp = subprocess_to_lp_list[scenario_idx]
        
        # Déterminer si c'est le scénario de référence
        is_reference = (scenario_idx == reference_idx)
        reference_for_display = None if is_reference else reference_data
        
        # Créer le graphique
        _, ax = create_stacked_bar_chart(
            scenario_data, 
            seuils_df, 
            subprocess_to_lp,
            scenario_name=scenario_name,
            reference_data=reference_for_display,
            ax=ax,
            show_legend=False  # Pas de légende locale
        )
        
        # Récupérer les handles/labels une seule fois (du premier subplot)
        if all_handles is None:
            handles, labels = ax.get_legend_handles_labels()
            # Supprimer les doublons
            by_label = dict(zip(labels, handles))
            all_handles = list(by_label.values())
            all_labels = list(by_label.keys())
    
    # Créer une légende globale unique en bas
    fig.legend(all_handles, all_labels, loc='lower center', 
              ncol=5, fontsize=10, bbox_to_anchor=(0.5, -0.02),
              frameon=True, fancybox=True, shadow=True)
    
    # Titre global
    fig.suptitle(
        f"Décomposition sectorielle des empreintes environnementales - Comparaison des scénarios\n"
        f"(Critère d'équité: {THRESHOLD_PARAM})",
        fontsize=16, fontweight='bold', y=0.98
    )
    
    plt.tight_layout(rect=[0, 0.03, 1, 0.96])
    return fig, axes_list


In [78]:
# ============================================================================
# 6. FONCTION PRINCIPALE - MULTI-SCÉNARIOS
# ============================================================================

def main():
    """Orchestration du script complet pour tous les scénarios"""
    
    print("=" * 80)
    print("ANALYSE MULTI-SCÉNARIOS - LIMITES PLANÉTAIRES")
    print("=" * 80)
    print()
    
    # Chargement des fichiers (identiques pour tous les scénarios)
    print("▶ Étape 1: Chargement des données sources communes")
    print("-" * 80)
    facteurs_carac_df = load_facteurs_carac()
    bridge_matrices_df = load_bridge_matrices()
    seuils_df = load_seuils()
    print()
    
    print(f"▶ Étape 2: Traitement de {len(SCENARIO_FOLDERS)} scénarios")
    print("-" * 80)
    
    all_scenarios_data = []
    all_subprocess_to_lp = []
    
    for scenario_idx, scenario_folder in enumerate(SCENARIO_FOLDERS):
        print(f"\n  Scénario {scenario_idx + 1}/{len(SCENARIO_FOLDERS)}: {scenario_folder.name}")
        
        # Traiter ce scénario
        data_by_subprocess, subprocess_to_lp = process_scenario(
            scenario_folder,
            facteurs_carac_df,
            bridge_matrices_df,
            seuils_df
        )
        
        if data_by_subprocess:
            all_scenarios_data.append(data_by_subprocess)
            all_subprocess_to_lp.append(subprocess_to_lp)
            print(f"    ✓ {len(data_by_subprocess)} sous-processus traités")
            
            # Générer et sauvegarder la figure individuelle pour ce scénario
            print(f"    Génération de la figure individuelle...")
            fig, ax = create_stacked_bar_chart(
                data_by_subprocess, 
                seuils_df, 
                subprocess_to_lp,
                scenario_name=f"Scénario: {scenario_folder.name}"
            )
            
            output_path = Path(BASE_DIR).parent / f"scenario_{scenario_idx:02d}_{scenario_folder.name}.png"
            fig.savefig(output_path, dpi=300, bbox_inches='tight')
            print(f"    ✓ Graphique sauvegardé: {output_path.name}")
            plt.close(fig)
        else:
            print(f"    ⚠ Aucune donnée valide pour ce scénario")
    
    print()
    print("=" * 80)
    
    # Vérification
    if len(all_scenarios_data) == len(SCENARIO_FOLDERS):
        print(f"✓ Tous les {len(all_scenarios_data)} scénarios ont été traités avec succès !")
    else:
        print(f"⚠ {len(all_scenarios_data)}/{len(SCENARIO_FOLDERS)} scénarios traités")
    
    print()
    print("▶ Étape 3: Génération de la figure de synthèse")
    print("-" * 80)
    
    # Créer la figure de synthèse
    if len(all_scenarios_data) == len(SCENARIO_FOLDERS):
        fig_synthesis, axes = create_synthesis_figure(
            all_scenarios_data,
            seuils_df,
            all_subprocess_to_lp,
            SCENARIO_NAMES
        )
        
        output_path_synthesis = Path(BASE_DIR).parent / "synthesis_all_scenarios.png"
        fig_synthesis.savefig(output_path_synthesis, dpi=300, bbox_inches='tight')
        print(f"  ✓ Figure de synthèse sauvegardée: {output_path_synthesis.name}")
        plt.close(fig_synthesis)
    else:
        print("  ⚠ Impossible de générer la synthèse (données manquantes)")
    
    print()
    print("=" * 80)
    print("Analyse terminée avec succès !")
    print("=" * 80)


if __name__ == "__main__":
    main()


ANALYSE MULTI-SCÉNARIOS - LIMITES PLANÉTAIRES

▶ Étape 1: Chargement des données sources communes
--------------------------------------------------------------------------------
Chargement des facteurs de caractérisation...
  ✓ 51 lignes chargées
Chargement de la matrice bridge...
  ✓ Matrice de shape (1001, 11)
Chargement des seuils...
  ✓ 13 lignes chargées

▶ Étape 2: Traitement de 6 scénarios
--------------------------------------------------------------------------------

  Scénario 1/6: 0. base-year_2015
  Traitement du scénario : 0. base-year_2015
    ✓ 7 sous-processus traités
    Génération de la figure individuelle...
    ✓ Graphique sauvegardé: scenario_00_0. base-year_2015.png

  Scénario 2/6: 1. TEND_2050
  Traitement du scénario : 1. TEND_2050
    ✓ 7 sous-processus traités
    Génération de la figure individuelle...
    ✓ Graphique sauvegardé: scenario_01_1. TEND_2050.png

  Scénario 3/6: 2. S1_2050
  Traitement du scénario : 2. S1_2050
    ✓ 7 sous-processus traités
  